# VGG11: physical nonidealities

Evaluate a CIFAR-10 VGG11-BN checkpoint with digital PyTorch, ideal-device CIM, and noisy CIM. Scan physical effects, their combinations, and quantization/read granularity.

Install MemFast, torchvision, NumPy, Matplotlib, and Jupyter. Set `MEMFAST_CIFAR10_ROOT` to an extracted torchvision-compatible CIFAR-10 directory and `MEMFAST_VGG_CHECKPOINT` to `cifar10_vgg11_bn-eaeebf42.pt` from the pytorch-cifar-models VGG release. The checkpoint must match `cifar_model('vgg11_bn', num_classes=10)`. The default is the first 512 test images; set `MEMFAST_VGG_SAMPLES=0` for all 10,000.

In [ ]:
import gc
import json
import os
import time
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

from memfast import Config, Nonidealities, Simulator, convert_model
from memfast.models import cifar_model

SEED = 135
torch.manual_seed(SEED)
torch.set_num_threads(4)
device = torch.device(
    os.environ.get("MEMFAST_NOTEBOOK_DEVICE", "cuda:0" if torch.cuda.is_available() else "cpu")
)
data_root = Path(os.environ.get("MEMFAST_CIFAR10_ROOT", "data/cifar10"))
checkpoint = Path(
    os.environ.get(
        "MEMFAST_VGG_CHECKPOINT",
        str(Path(torch.hub.get_dir()) / "checkpoints/cifar10_vgg11_bn-eaeebf42.pt"),
    )
)
max_samples = int(os.environ.get("MEMFAST_VGG_SAMPLES", "512"))
batch_size = 8
if max_samples < 0:
    raise ValueError("Use a positive sample limit or 0 for the full test split")
if not checkpoint.is_file():
    raise FileNotFoundError(
        f"Set MEMFAST_VGG_CHECKPOINT to the real VGG11-BN checkpoint: {checkpoint}"
    )
print("Device:", device, "sample limit:", max_samples or "full test split")

## 1. Load fixed inputs and a real checkpoint

Use the repository's CIFAR evaluation preprocessing: RGB channel means (0.4914, 0.4822, 0.4465) and standard deviations (0.2023, 0.1994, 0.2010). No augmentation or shuffling is used. Every configuration sees the same ordered batches. VGG's batch normalization, ReLU, pooling, dropout in evaluation mode, and biases remain digital; every convolution and linear matrix is mapped.

In [ ]:
preprocess = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ]
)
dataset = datasets.CIFAR10(data_root, train=False, download=False, transform=preprocess)
sample_indices = list(range(len(dataset) if max_samples == 0 else min(max_samples, len(dataset))))
batches = list(DataLoader(Subset(dataset, sample_indices), batch_size=batch_size, shuffle=False))
targets = torch.cat([labels for _, labels in batches])
model = cifar_model("vgg11_bn", num_classes=10).float().eval()
state = torch.load(checkpoint, map_location="cpu", weights_only=True)
model.load_state_dict(state, strict=True)
layer_names = [
    name for name, layer in model.named_modules() if isinstance(layer, (nn.Conv2d, nn.Linear))
]
print("Checkpoint:", checkpoint.name)
print("Test examples:", len(targets), "mapped layers:", len(layer_names))
print("Layers:", ", ".join(layer_names))


@torch.inference_mode()
def evaluate(network):
    network.eval()
    logits = torch.cat([network(images.to(device)).cpu() for images, _ in batches])
    assert logits.shape == (len(targets), 10) and torch.isfinite(logits).all()
    return dict(
        count=len(targets),
        correct=int((logits.argmax(1) == targets).sum()),
        top1=float((logits.argmax(1) == targets).float().mean()),
        cross_entropy=float(nn.functional.cross_entropy(logits, targets)),
        logits=logits,
    )


model.to(device)
digital_result = evaluate(model)
model.cpu()
print(
    f"Digital top-1: {100 * digital_result['top1']:.2f}% ({digital_result['correct']}/{len(targets)})"
)

## 2. From signed bits to a physical read

Each signed integer is reconstructed from slices with factors $s_j$; the sign slice has a negative factor. For a weight-slice state $q_w$ with maximum $m_w=2^{b_w}-1$ and input-slice state $q_a$ with maximum $m_a=2^{b_a}-1$,

$$G_0=G_{\min}+(G_{\max}-G_{\min})q_w/m_w,\qquad V_0=V_{\mathrm{read}}q_a/m_a.$$

Programming variation uses multiplicative lognormal conductance noise:

$$G_{\mathrm{program}}=G_0 e^{\sigma_w z_{\mathrm{cell}}},\qquad G_{\mathrm{drift}}=G_{\mathrm{program}}(1+t/t_{\mathrm{ref}})^{-\nu}.$$

A fixed per-cell uniform sample subsequently forces fractions `stuck_at_low` and `stuck_at_high` to the corresponding conductance endpoints. Their probabilities must sum to at most one. In shared read mode, $G'=G_{\mathrm{stuck}}e^{\sigma_r z_{\mathrm{cell,event}}}$. Voltage variation multiplies the input voltage by $1+\sigma_v z_{\mathrm{vector,array,input\ slice,row,event}}$.

For each physical subread of $\ell$ rows, MemFast subtracts the nominal low-conductance floor and forms

$$I=\sum_{k=1}^{\ell}V'_k(G'_k-G_{\min})+\sigma_o I_{\mathrm{unit}}\ell z_{\mathrm{output}},\quad I_{\mathrm{unit}}=(G_{\max}-G_{\min})V_{\mathrm{read}}.$$

Output noise is independently sampled per vector, physical subread, output column, input/weight slice pair, and event. It is a current perturbation before ADC conversion. Let $u=m_a m_w I/I_{\mathrm{unit}}$ and $M=2^{b_{\mathrm{ADC}}}-1$. The implemented nonlinear ADC is

$$Q(I)=\operatorname{round}\left[\operatorname{clip}\left(\frac{u}{1+|\eta u/M|},0,M\right)\right].$$

Signed digital reconstruction sums $s_a s_w Q(I)$ and restores the input/weight quantization scales separately for every subread before summing partitions. A low ADC width can therefore clip ideal slice currents too. With binary slices and 64 rows, 7 bits cover counts 0..64; the package default of 6 bits stops at 63. We keep the default as an explicit baseline and include 8-bit comparisons.

### Sampling and scale granularity

| Setting | Meaning |
| --- | --- |
| `write_variation` / stuck faults | Fixed per physical cell, weight slice, matrix name, and seed; shared across vectors and reads. |
| `read_mode='shared'` | A new cell field per forward event, shared across vectors and input slices within that event. |
| `read_mode='slice_pair'` | A new cell field per event and input/weight slice pair, shared across vectors. |
| `read_mode='output'` | A Gaussian current approximation with the lognormal mean/variance, independently sampled per output/read/slice pair/vector/event; not explicit per-vector cell sampling. |
| `input_quant_gran=(1, None)` | One symmetric INT scale per input vector, across its entire reduction dimension. A convolution vector is an unfolded image patch. |
| `weight_quant_gran=(None, 1)` | One scale per output channel over the reduction dimension. Physical weights have orientation `[K, O]`. |
| `(read_rows, 1)` / `(1, read_rows)` | Weight/input scales local to a read-aligned reduction block. These are scale groups, not slice widths. |

For signed INT, $s=\max|x|/(2^{b-1}-1)$ and $q=\operatorname{clip}(\operatorname{round}(x/s),-(2^{b-1}-1),2^{b-1}-1)$. Zero groups use a finite fallback scale. Quantization group boundaries must align with physical ADC reads. Default slices are binary; their layouts run from sign to least significant slice when supplied explicitly.

In `output` read mode, the unnormalized-current mean shift is $(e^{\sigma_r^2/2}-1)\sum V_kG_k$, and variance is $e^{\sigma_r^2}(e^{\sigma_r^2}-1)\sum V_k^2G_k^2$. The nominal floor is still subtracted in the deterministic current. This approximation and shared-cell modes have different correlations; the same numerical sigma does not make them identical experiments.

One model forward is one event per matrix, so event-dependent noise depends on batch grouping. Keep the seed, sample order and batch size fixed across comparisons.

In [ ]:
base_config = Config(
    activation_bits=8,
    weight_bits=8,
    array_size=(64, 64),
    adc_bits=6,
    input_quant_gran=(1, None),
    weight_quant_gran=(None, 1),
    seed=SEED,
    nonidealities=Nonidealities(write_variation=0.02),
)
ideal_config = replace(base_config, nonidealities=Nonidealities(write_variation=0.0))
print("Default physical configuration:", asdict(base_config))

## 3. Declare the finite scan before evaluation

Three complementary scans keep the exercise bounded. Every point maps all 11 eligible layers from the same original checkpoint with fixed execution scheduling; there is no automatic accuracy-based selection.

1. **Physical effects:** ideal and default baselines; write sigma {0, 0.02, 0.05, 0.10, 0.20}; read sigma {0.02, 0.05, 0.10} in each of three modes; voltage sigma {0.02, 0.05, 0.10}; output-current sigma {0.002, 0.01, 0.02}; ADC nonlinearity {0.02, 0.05, 0.10}; each stuck probability {0.001, 0.01, 0.05}; drift coefficient {0.01, 0.03, 0.06} with t=1000 s and reference t=1 s. Except for the explicitly ideal/write points, these vary one field around write=0.02.
2. **Joint write/read/ADC:** the complete 3 x 2 x 3 grid of write {0, 0.02, 0.08}, shared read {0, 0.02}, and ADC {5, 6, 8}.
3. **Geometry and scales:** the complete 3 x 2 x 2 grid of physical rows {32, 64, 128}, rows per read {half, full}, and scale scheme {per-vector/per-channel, read-local}. Columns=64, ADC=8, and write=0.02 stay fixed.

Repeated identical configurations share one evaluation while retaining all group memberships. The following cell prints the declared and unique configuration counts.

In [ ]:
cases = []
case_keys = {}
declarations = []


def add_case(group, label, config):
    key = json.dumps(asdict(config), sort_keys=True)
    if key not in case_keys:
        case_keys[key] = len(cases)
        cases.append(dict(label=label, config=config, groups=[], aliases=[]))
    case = cases[case_keys[key]]
    if group not in case["groups"]:
        case["groups"].append(group)
    case["aliases"].append(label)
    declarations.append(dict(group=group, label=label, index=case_keys[key]))


add_case("effects", "ideal ADC6", ideal_config)
add_case("effects", "default write0.02 ADC6", base_config)
for value in (0.0, 0.02, 0.05, 0.10, 0.20):
    add_case(
        "effects",
        f"write={value:g}",
        replace(base_config, nonidealities=Nonidealities(write_variation=value)),
    )
for mode in ("shared", "slice_pair", "output"):
    for value in (0.02, 0.05, 0.10):
        effect = replace(base_config.nonidealities, read_variation=value, read_mode=mode)
        add_case("effects", f"read {mode}={value:g}", replace(base_config, nonidealities=effect))
for field, values in (
    ("voltage_variation", (0.02, 0.05, 0.10)),
    ("output_noise", (0.002, 0.01, 0.02)),
    ("adc_nonlinearity", (0.02, 0.05, 0.10)),
    ("stuck_at_low", (0.001, 0.01, 0.05)),
    ("stuck_at_high", (0.001, 0.01, 0.05)),
    ("drift_coefficient", (0.01, 0.03, 0.06)),
):
    for value in values:
        changes = {field: value}
        if field == "drift_coefficient":
            changes.update(drift_time=1000.0, drift_reference_time=1.0)
        effect = replace(base_config.nonidealities, **changes)
        add_case("effects", f"{field}={value:g}", replace(base_config, nonidealities=effect))
for write in (0.0, 0.02, 0.08):
    for read in (0.0, 0.02):
        for adc in (5, 6, 8):
            effect = Nonidealities(write_variation=write, read_variation=read, read_mode="shared")
            add_case(
                "joint",
                f"write{write:g}/read{read:g}/ADC{adc}",
                replace(base_config, adc_bits=adc, nonidealities=effect),
            )
for rows in (32, 64, 128):
    for read_rows in (rows // 2, rows):
        for local_scales in (False, True):
            config = replace(
                base_config,
                array_size=(rows, 64),
                rows_per_read=read_rows,
                adc_bits=8,
                input_quant_gran=(1, read_rows if local_scales else None),
                weight_quant_gran=(read_rows if local_scales else None, 1),
            )
            add_case("geometry", f"R{rows}/read{read_rows}/local={local_scales}", config)
for group in ("effects", "joint", "geometry"):
    print(
        group,
        "declared:",
        sum(item["group"] == group for item in declarations),
        "unique within group:",
        len({item["index"] for item in declarations if item["group"] == group}),
    )
print("Total unique CIM evaluations:", len(cases), "plus one digital evaluation")

## 4. Execute and keep the actual outputs

Run the declared scan on CPU or CUDA. Each conversion resets the matrix read events. Results retain logits in memory for inspection; elapsed time covers conversion, transfers and inference.

In [ ]:
scan_results = []
for index, case in enumerate(cases):
    started = time.perf_counter()
    engine = Simulator(case["config"], device=str(device))
    analog = convert_model(model, engine)
    result = evaluate(analog)
    result.update(
        label=case["label"],
        config=asdict(case["config"]),
        groups=case["groups"],
        aliases=case["aliases"],
        seconds=time.perf_counter() - started,
        agreement=float(
            (result["logits"].argmax(1) == digital_result["logits"].argmax(1)).float().mean()
        ),
    )
    scan_results.append(result)
    print(
        f"{index + 1:02d}/{len(cases)} {case['label']:<38} top1={100 * result['top1']:6.2f}% agreement={100 * result['agreement']:6.2f}%",
        flush=True,
    )
    del analog, engine
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
assert len(scan_results) == len(cases)
assert all(result["count"] == len(targets) for result in scan_results)

## 5. Inspect accuracy, interactions, and granularity

Top-1 compares predictions with the dataset labels. Prediction agreement compares CIM predictions with the digital checkpoint. Rows per read determines where ADC conversion occurs; scale groups determine which operands share a quantization scale.

In [ ]:
effect_results = [item for item in scan_results if "effects" in item["groups"]]
fig, ax = plt.subplots(figsize=(11, 10), constrained_layout=True)
positions = np.arange(len(effect_results))
ax.barh(positions, [100 * item["top1"] for item in effect_results], color="#167c80")
ax.set_yticks(positions, [item["label"] for item in effect_results])
ax.invert_yaxis()
ax.axvline(100 * digital_result["top1"], color="#a83246", linestyle="--", label="Digital")
ax.set(xlabel=f"Top-1 accuracy (%) on {len(targets)} fixed test images", xlim=(0, 100))
ax.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)
joint = [item for item in declarations if item["group"] == "joint"]
joint_values = np.array([100 * scan_results[item["index"]]["top1"] for item in joint]).reshape(6, 3)
image = axes[0].imshow(joint_values, vmin=0, vmax=100, cmap="viridis", aspect="auto")
axes[0].set_xticks(range(3), ["ADC 5", "ADC 6", "ADC 8"])
axes[0].set_yticks(
    range(6), [f"write {w:g}, read {r:g}" for w in (0, 0.02, 0.08) for r in (0, 0.02)]
)
axes[0].set_title("Complete bounded joint grid")
for row in range(6):
    for column in range(3):
        axes[0].text(
            column,
            row,
            f"{joint_values[row, column]:.1f}",
            ha="center",
            va="center",
            color="black" if joint_values[row, column] > 55 else "white",
        )
fig.colorbar(image, ax=axes[0], label="Top-1 (%)")
geometry = [item for item in declarations if item["group"] == "geometry"]
axes[1].barh(
    range(len(geometry)),
    [100 * scan_results[item["index"]]["top1"] for item in geometry],
    color="#bc6b37",
)
axes[1].set_yticks(range(len(geometry)), [item["label"] for item in geometry])
axes[1].invert_yaxis()
axes[1].axvline(100 * digital_result["top1"], color="#a83246", linestyle="--")
axes[1].set(xlabel="Top-1 (%)", xlim=(0, 100), title="Read boundaries and scale groups")
plt.show()

### Observed execution

Results for **59 unique CIM configurations** use the checkpoint above, test indices 0..511, batch size 8, seed 135, signed A8/W8, and default scheduling. The physical-effects group contains 32 unique settings, the joint grid has 18 points, and geometry/scales has 12; overlapping configurations are evaluated once.

| Configuration | Correct / 512 | Top-1 (%) |
| --- | --- | --- |
| Digital checkpoint | 481 | 93.95 |
| Ideal devices, ADC6 | 482 | 94.14 |
| Default write 0.02, ADC6 | 482 | 94.14 |
| Write 0.20, ADC6 | 340 | 66.41 |
| Output-current noise 0.01, write 0.02, ADC6 | 55 | 10.74 |
| Drift coefficient 0.06, t=1000 s, write 0.02, ADC6 | 115 | 22.46 |

The 18 joint-grid scores range from 93.36% to 94.14%; the 12 geometry/scale scores range from 93.95% to 94.14%. Edit the parameter grids above to compare your own device settings.